# List and Filter

List the objects of a folder, recursively or one level at a time, take only what you need, filter by any attribute, and count. Listing is lazy: S3 is called page by page while you iterate.

## Setup

This cell only prepares the notebook: it connects to the S3 bucket that the s3pathlib CI uses and empties the folder of this chapter, so every run starts from the same state. In your own code you only need the two lines marked below, with your own AWS profile and bucket.
`show_error` prints an exception in one line, so the demos can show what goes wrong.

In [1]:
import os

import boto3
from s3pathlib.api import S3Path
from s3pathlib.api import s3_context

boto_ses = boto3.Session(profile_name=os.environ["S3PATHLIB_TEST_AWS_PROFILE"])
s3_context.attach_s3_client(boto_ses.client("s3"))  # <-- in your code
aws_account_id = boto_ses.client("sts").get_caller_identity()["Account"]
bucket = f"{aws_account_id}-{boto_ses.region_name}-s3pathlib-test"  # <-- your bucket

# every demo of this chapter writes under this folder
s3dir_chapter = S3Path(bucket, "projects/s3pathlib/docs/04-List-and-Filter/")
_ = s3dir_chapter.delete()


def show_error(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

All demos of this chapter list the same small tree. `hard-folder/` has a folder marker (an empty object `hard-folder/`, what `mkdir` and the S3 console make); `soft-folder/` has none, it only exists because keys start with it.

In [2]:
s3dir = s3dir_chapter / "tree/"
files = {
    "README.txt": 100,
    "data-2024.csv": 2000,
    "hard-folder/1.txt": 300,
    "soft-folder/2.csv": 4000,
    "soft-folder/sub/3.csv": 500,
    "soft-folder/sub/4.json": 60,
}
for key, size in files.items():
    _ = (s3dir / key).write_bytes(b"x" * size)
# the folder already "exists" (a key starts with it), so exist_ok=True
(s3dir / "hard-folder/").mkdir(exist_ok=True)

## 1. `iter_objects`: every object under a folder

By default it is recursive. The paths come in the order S3 returns them, sorted by key as a string (so `data-2024.csv` comes before `hard-folder/...`, because `-` is before `/`). Folder markers are skipped.

In [3]:
[p - s3dir for p in s3dir.iter_objects()]

[S3RelPath('README.txt'),
 S3RelPath('data-2024.csv'),
 S3RelPath('hard-folder/1.txt'),
 S3RelPath('soft-folder/2.csv'),
 S3RelPath('soft-folder/sub/3.csv'),
 S3RelPath('soft-folder/sub/4.json')]

`recursive=False` keeps the objects directly in the folder:

In [4]:
[p - s3dir for p in s3dir.iter_objects(recursive=False)]

[S3RelPath('README.txt'), S3RelPath('data-2024.csv')]

## 2. `iterdir`: one level, folders included

Like `pathlib.Path.iterdir`: the files and the folders directly in this folder, with or without a marker, mixed in key order.

In [5]:
[p - s3dir for p in s3dir.iterdir()]

[S3RelPath('README.txt'),
 S3RelPath('data-2024.csv'),
 S3RelPath('hard-folder/'),
 S3RelPath('soft-folder/')]

In [6]:
[p - s3dir for p in (s3dir / "soft-folder/").iterdir()]

[S3RelPath('soft-folder/2.csv'), S3RelPath('soft-folder/sub/')]

Both need a folder. Without the trailing `/` the path is a file, and listing it raises before calling AWS (to list keys by a string prefix, use `better_client.paginate_list_objects_v2`, see section 7):

In [7]:
show_error(lambda: S3Path(bucket, "projects/s3pathlib/docs/04-List-and-Filter/tree").iter_objects())

S3PathIsNotFolderError: 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/04-List-and-Filter/tree' is not a dir!


## 3. Take what you need

`iter_objects` and `iterdir` return an `S3PathIterProxy`: an iterator with a few helpers. Pages are fetched while iterating, so `one()` or `many(2)` on a folder with millions of objects only makes one API call.

In [8]:
print(s3dir.iter_objects().one() - s3dir)
print([p - s3dir for p in s3dir.iter_objects().many(2)])
print([p - s3dir for p in s3dir.iter_objects().skip(4).all()])
print((s3dir / "hard-folder/").iterdir().one_or_none() - s3dir)
print((s3dir / "empty/").iter_objects().one_or_none())

S3RelPath('README.txt')
[S3RelPath('README.txt'), S3RelPath('data-2024.csv')]
[S3RelPath('soft-folder/sub/3.csv'), S3RelPath('soft-folder/sub/4.json')]
S3RelPath('hard-folder/1.txt')
None


`limit` stops after that many paths; `batch_size` (1 to 1000, default 1000) is how many keys one API call asks for. An `S3PathIterProxy` can be iterated **only once**, like any iterator; call `iter_objects()` again, or keep `.all()` in a list.

In [9]:
proxy = s3dir.iter_objects(limit=3, batch_size=2)
print(len(proxy.all()))
print(proxy.all())

3
[]


## 4. Filter

`filter_by_ext` keeps the files with one of the extensions (case insensitive):

In [10]:
[p - s3dir for p in s3dir.iter_objects().filter_by_ext(".csv", ".json")]

[S3RelPath('data-2024.csv'),
 S3RelPath('soft-folder/2.csv'),
 S3RelPath('soft-folder/sub/3.csv'),
 S3RelPath('soft-folder/sub/4.json')]

`filter` takes conditions written on the class attributes, like an ORM: `S3Path.size >= 1000` is a function that tests one path. Several conditions in one `filter` (or chained `filter` calls) must all be true; `and_`, `or_` and `not_` combine them in other ways. The filters run in memory, on the paths S3 returns.

In [11]:
from s3pathlib.api import and_
from s3pathlib.api import or_
from s3pathlib.api import not_

print([p - s3dir for p in s3dir.iter_objects().filter(S3Path.size >= 1000)])
print([p - s3dir for p in s3dir.iter_objects().filter(S3Path.ext == ".csv", S3Path.size < 1000)])
print([p - s3dir for p in s3dir.iter_objects().filter(or_(S3Path.ext == ".txt", S3Path.ext == ".json"))])
print([p - s3dir for p in s3dir.iter_objects().filter(not_(S3Path.dirname == "sub"))])

[S3RelPath('data-2024.csv'), S3RelPath('soft-folder/2.csv')]
[S3RelPath('soft-folder/sub/3.csv')]
[S3RelPath('README.txt'), S3RelPath('hard-folder/1.txt'), S3RelPath('soft-folder/sub/4.json')]
[S3RelPath('README.txt'), S3RelPath('data-2024.csv'), S3RelPath('hard-folder/1.txt'), S3RelPath('soft-folder/2.csv')]


Every comparison works (`==`, `!=`, `<`, `<=`, `>`, `>=`), and so do `between(lower, upper)`, `startswith`, `endswith` and `contains`. The filterable attributes are `bucket`, `key`, `uri`, `arn`, `parts`, `basename`, `dirname`, `fname`, `ext`, `abspath`, `dirpath`, and the metadata `size`, `etag`, `last_modified_at`, `version_id`, `expire_at`.

In [12]:
from datetime import datetime, timedelta, timezone

one_hour_ago = datetime.now(timezone.utc) - timedelta(hours=1)
print([p - s3dir for p in s3dir.iter_objects().filter(S3Path.size.between(100, 500))])
print([p - s3dir for p in s3dir.iter_objects().filter(S3Path.basename.startswith("data"))])
print([p - s3dir for p in s3dir.iter_objects().filter(S3Path.parts.contains("sub"))])
print(len(s3dir.iter_objects().filter(S3Path.last_modified_at >= one_hour_ago).all()))

[S3RelPath('README.txt'), S3RelPath('hard-folder/1.txt'), S3RelPath('soft-folder/sub/3.csv')]
[S3RelPath('data-2024.csv')]
[S3RelPath('soft-folder/sub/3.csv'), S3RelPath('soft-folder/sub/4.json')]
6


Any function that takes a path and returns a `bool` works too:

In [13]:
[p - s3dir for p in s3dir.iter_objects().filter(lambda p: len(p.fname) == 1)]

[S3RelPath('hard-folder/1.txt'),
 S3RelPath('soft-folder/2.csv'),
 S3RelPath('soft-folder/sub/3.csv'),
 S3RelPath('soft-folder/sub/4.json')]

## 5. Listed paths already know their size

Each listed path carries what the list call returned (`size`, `etag`, `last_modified_at`), so filtering on them costs no extra call. Other fields, such as the user `metadata`, are loaded with a `head_object` call the first time you read them; see [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb).

In [14]:
p = s3dir.iter_objects().one()
p.size, p.etag

(100, 'aed563ecafb4bcc5654c597a421547b2')

## 6. Count and total size

`count_objects` and `calculate_total_size` go through every object under the folder (recursively), like "Calculate total size" in the S3 console. Folder markers only count with `include_folder=True`.

In [15]:
print(s3dir.count_objects())
print(s3dir.count_objects(include_folder=True))
print(s3dir.calculate_total_size())
print(s3dir.calculate_total_size(for_human=True))

6
7
(6, 6960)
(6, '6.80 KB')


## 7. Keys that `S3Path` cannot represent

S3 allows keys like `a//b.txt` (two `/` in a row) or `/a.txt` (a leading `/`), but a path collapses repeated `/`, so the `S3Path` of `a//b.txt` would be `a/b.txt`: another object. Rather than give you a path to the wrong object, listing such a key raises `S3ValidationError`. List them with `better_client.paginate_list_objects_v2`, which returns the raw boto3 responses and also lists by any string prefix.

In [16]:
from s3pathlib.api import better_client

s3dir_odd = s3dir_chapter / "odd-keys/"
s3_context.s3_client.put_object(Bucket=bucket, Key=s3dir_odd.key + "a//b.txt", Body=b"odd")
show_error(lambda: s3dir_odd.iter_objects().all())

result = better_client.paginate_list_objects_v2(s3_context.s3_client, bucket, prefix=s3dir_odd.key)
[content["Key"].removeprefix(s3dir_odd.key) for content in result.contents()]

S3ValidationError: the S3 key 'projects/s3pathlib/docs/04-List-and-Filter/odd-keys/a//b.txt' in bucket '123456789012-us-east-1-s3pathlib-test' cannot be represented by S3Path, which joins consecutive '/' and drops a leading '/' (its path would be 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/04-List-and-Filter/odd-keys/a/b.txt', another object). List this folder with s3pathlib.api.better_client or boto3 instead


['a//b.txt']

Methods that work on a whole folder (`delete`, `copy_dir`, `move_to`) work on the keys S3 returns, not on paths, so they handle such keys:

In [17]:
_ = s3dir_odd.delete()
s3dir_odd.exists()

False

## Common pitfalls

- **The trailing slash.** `iter_objects` and `iterdir` need a folder path.
- **Iterate once.** An `S3PathIterProxy` is consumed by the first `.all()` or `for` loop.
- **S3 order is not sorted order.** Results come in key string order; `sorted(paths)` puts each folder before its content (see [Path Manipulation](../02-Path-Manipulation/index.ipynb)).
- **Filters run on your side.** S3 only filters by prefix; a filter on `size` still lists every key under the folder. Narrow the folder first.

## See also

- [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb): what a listed path knows, and what triggers a `head_object`.
- [Versioning](../07-Versioning/index.ipynb): `list_object_versions`.